<a href="https://colab.research.google.com/github/alfnihilus/computational-chem/blob/main/self-study/creat_molecul.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# First, install the PubChemPy and RDKit Python libraries
!pip install pubchempy rdkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.7/36.7 MB 33.6 MB/s eta 0:00:00


In [27]:
!pip install pyopsin

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 70.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 495.9/495.9 kB 40.6 MB/s eta 0:00:00
  Created wheel for pyopsin: filename=pyopsin-0.2.4-py3-none-any.whl size=12162098 sha256=254bd03b5437777d7c1c00d4e0e9c4c1b6a11ecb2e3ceac71ed9dbd754499da9
  Stored in directory: /root/.cache/pip/wheels/66/fd/4e/cfaa77ff8a6f1b56a9e25f6daca935f25b6555e3a12c42e909
Successfully built pyopsin


In [44]:
import pubchempy as pcp

# @markdown # Pembuatan SMILES
# @markdown ### Masukkan Nama IUPAC atau Nama Umum/Trivial (Cth. Etanol, 2,4,6-trinitrotoluene)
input_nama_senyawa = "caffein" # @param {type:"string"}

def cek_smiles_lengkap(nama_atau_rumus):
    results = pcp.get_compounds(nama_atau_rumus, 'name') # atau 'formula'

    if not results:
        print(f"Tidak ada hasil yang ditemukan untuk: {nama_atau_rumus}")
        return

    for compound in results:
        print(f"Nama: {compound.iupac_name}")
        # Memanggil dua jenis SMILES yang berbeda
        print(f"Canonical SMILES: {compound.canonical_smiles}")
        print(f"Isomeric SMILES : {compound.isomeric_smiles}")
        print("-" * 30)

# Memanggil fungsi dengan input dari pengguna
cek_smiles_lengkap(input_nama_senyawa)


Nama: 1,3,7-trimethylpurine-2,6-dione
Canonical SMILES: CN1C=NC2=C1C(=O)N(C(=O)N2C)C
Isomeric SMILES : CN1C=NC2=C1C(=O)N(C(=O)N2C)C
------------------------------


/tmp/ipykernel_369/195278257.py:17: PubChemPyDeprecationWarning: canonical_smiles is deprecated: Use connectivity_smiles instead
  print(f"Canonical SMILES: {compound.canonical_smiles}")
/tmp/ipykernel_369/195278257.py:18: PubChemPyDeprecationWarning: isomeric_smiles is deprecated: Use smiles instead
  print(f"Isomeric SMILES : {compound.isomeric_smiles}")


In [45]:
import pubchempy as pcp
from rdkit import Chem
from rdkit.Chem import AllChem

def generate_molecule(identifier):
    try:
        results = []
        search_type = ""

        # 1. Coba cari berdasarkan CID jika identifier adalah angka murni
        if identifier.isdigit():
            print(f"Mencari sebagai CID: {identifier}...")
            results = pcp.get_compounds(identifier, 'cid')
            if results:
                search_type = "CID"

        # 2. Jika belum ditemukan, coba cari sebagai SMILES
        #    SMILES seringkali mengandung '=', '#', '(', ')', '[', ']', '.', '@', '/', '\\'
        #    dan tidak boleh hanya angka. Periksa beberapa karakter khas SMILES.
        if not results and (any(c in identifier for c in ['=', '#', '(', ')', '[', ']', '.', '@', '/', '\\']) or (not identifier.isalpha() and not identifier.isdigit())):
            print(f"Tidak ditemukan sebagai CID. Mencoba sebagai SMILES: {identifier}...")
            results = pcp.get_compounds(identifier, 'smiles')
            if results:
                search_type = "SMILES"

        # 3. Jika belum ditemukan, coba cari sebagai Nama
        if not results:
            print(f"Tidak ditemukan sebagai CID/SMILES. Mencoba sebagai Nama: {identifier}...")
            results = pcp.get_compounds(identifier, 'name')
            if results:
                search_type = "Nama"

        # 4. Jika belum ditemukan, coba cari berdasarkan Rumus
        if not results:
            print(f"Tidak ditemukan sebagai CID/SMILES/Nama. Mencoba sebagai Rumus: {identifier}...")
            results = pcp.get_compounds(identifier, 'formula')
            if results:
                search_type = "Rumus"

        if not results:
            print("Molekul tidak ditemukan di database PubChem.")
            return

        # Mengambil hasil pertama yang ditemukan
        comp = results[0]
        smiles = comp.smiles
        print(f"Ditemukan! Tipe: {search_type}, SMILES: {smiles}")

        # 3. Proses RDKit untuk Koordinat 3D
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            print(f"Error: Tidak dapat mengonversi SMILES '{smiles}' menjadi objek molekul RDKit.")
            return

        mol = Chem.AddHs(mol)

        # Memberikan koordinat 3D awal
        # Menggunakan AllChem.ETKDGv2() untuk hasil yang lebih baik
        AllChem.EmbedMolecule(mol, AllChem.ETKDGv2())
        # Optimasi struktur agar stabil (Universal Force Field)
        AllChem.UFFOptimizeMolecule(mol)

        # 4. Cetak koordinat
        print(f"\nKoordinat untuk {identifier} (dari {search_type}):")
        conf = mol.GetConformer()
        for i, atom in enumerate(mol.GetAtoms()):
            pos = conf.GetAtomPosition(i)
            print(f"{atom.GetSymbol()} {pos.x:.4f} {pos.y:.4f} {pos.z:.4f}")
        print("\nAnda dapat menyalin koordinat di atas untuk penggunaan lebih lanjut.")

    except Exception as e:
        print(f"Error: {e}")

# Input dari user
# @markdown # Pembuatan Koordinat Secara Otomatis (Sumber PubChem)
# @markdown ### Masukkan Nama atau Rumus Kimia (misal: H2O, Ethanol, C6H6, Flavonoid, 2244, CCCCCCCCCCCC)
user_input ="" # @param {type:"string"}
generate_molecule(user_input)


Tidak ditemukan sebagai CID. Mencoba sebagai SMILES: 64-17-5...
Error: PubChem HTTP Error 400 PUGREST.BadRequest: Unable to standardize the given structure - perhaps some special characters need to be escaped or data packed in a MIME form? (error: , status: 400, output: Caught ncbi::CException: Standardization failed, Output Log:, Record 1: Warning: Cactvs Ensemble cannot be created from input string, Record 1: Error: Unable to convert input into a compound object, , )


In [3]:
# First, install the Ase Python libraries
!pip install ase

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 35.1 MB/s eta 0:00:00


In [20]:
import re
import numpy as np
from ase import Atoms
from ase.io import write

def parse_formula(formula):
    # Memisahkan rumus (misal U79O90C10 menjadi [('U', 79), ('O', 90), ('C', 10)])
    elements = re.findall(r'([A-Z][a-z]*)(\d*)', formula)
    parsed = []
    for el, count in elements:
        count = int(count) if count else 1
        parsed.extend([el] * count)
    return parsed

def generate_random_cluster(formula, density=0.5):
    atom_list = parse_formula(formula)
    num_atoms = len(atom_list)

    # Menghitung perkiraan ukuran kotak berdasarkan jumlah atom
    box_size = (num_atoms / density) ** (1/3)

    # Membuat koordinat acak (x, y, z)
    positions = np.random.rand(num_atoms, 3) * box_size

    # Membuat objek atom ASE
    cluster = Atoms(symbols=atom_list, positions=positions)

    # Optimasi jarak antar atom sederhana agar tidak bertumpukan (Push-apart)
    # Catatan: Ini bukan optimasi kimia, hanya agar tidak overlap di GaussView
    cluster.center(vacuum=5.0)

    # Cetak koordinat
    print(f"\nKoordinat acak untuk {formula} dengan {num_atoms} atom:")
    for i, symbol in enumerate(cluster.get_chemical_symbols()):
        pos = cluster.get_positions()[i]
        print(f"{symbol} {pos[0]:.4f} {pos[1]:.4f} {pos[2]:.4f}")
    print("\nAnda dapat menyalin koordinat di atas untuk penggunaan lebih lanjut.")

# Contoh penggunaan
# @markdown # Pembuatan Molekul Manual
# @markdown ### Masukkan rumus kimia acak (misal U10O20C5)
rumus ="CC1CCC1" # @param {type:"string"}
generate_random_cluster(rumus)


Koordinat acak untuk CC1CCC1 dengan 5 atom:
C 5.3584 5.2058 5.0000
C 5.5140 5.0975 6.7055
C 6.4055 5.1138 6.2049
C 5.0000 5.8554 6.8468
C 5.0835 5.0000 6.6386

Anda dapat menyalin koordinat di atas untuk penggunaan lebih lanjut.


In [21]:
import re
import numpy as np
from ase import Atoms

def parse_formula(formula):
    # Memisahkan rumus (misal U79O90C10 menjadi [('U', 79), ('O', 90), ('C', 10)])
    elements = re.findall(r'([A-Z][a-z]*)(\d*)', formula)
    parsed = []
    for el, count in elements:
        count = int(count) if count else 1
        parsed.extend([el] * count)
    return parsed

def generate_random_cluster(formula, density=0.2):
    atom_list = parse_formula(formula)
    num_atoms = len(atom_list)

    # Menghitung perkiraan ukuran kotak
    box_size = (num_atoms / density) ** (1/3)

    # Membuat koordinat acak (x, y, z) yang bisa negatif dan bertabrakan
    # Menggunakan (np.random.rand(num_atoms, 3) - 0.5) * box_size * 2
    # untuk menghasilkan angka antara -box_size dan +box_size
    positions = (np.random.rand(num_atoms, 3) - 0.5) * box_size * 2

    # Membuat objek atom ASE
    cluster = Atoms(symbols=atom_list, positions=positions)
    # Tidak lagi menggunakan cluster.center(vacuum=5.0) secara default
    # jika tujuannya adalah membiarkan koordinat negatif dan bertabrakan

    # Cetak koordinat sesuai format awal kamu
    print(f"\nKoordinat acak untuk {formula} dengan {num_atoms} atom:")
    for i, symbol in enumerate(cluster.get_chemical_symbols()):
        pos = cluster.get_positions()[i]
        print(f"{symbol} {pos[0]:.4f} {pos[1]:.4f} {pos[2]:.4f}")
    print("\nAnda dapat menyalin koordinat di atas untuk penggunaan lebih lanjut.")

# --- Bagian Input Colab ---
# @markdown # Pembuatan Molekul Manual
# @markdown ### Masukkan rumus kimia acak (misal U10O20C5)
rumus = "CC1CCC1" # @param {type:"string"}
generate_random_cluster(rumus)



Koordinat acak untuk CC1CCC1 dengan 5 atom:
C -2.6391 -0.1787 -0.6694
C -1.3797 0.9889 -1.2644
C 1.0324 0.3887 -1.0324
C 2.5018 2.2393 -0.9868
C -2.5210 0.5248 -1.8712

Anda dapat menyalin koordinat di atas untuk penggunaan lebih lanjut.
